<a target="_blank" href="https://colab.research.google.com/github/keatonkraiger/Whisper-Transcribe-and-Translate-Tutorial/blob/main/Whisper_Tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<a target="_blank" href="https://youtu.be/i4Sgg-ptRzs">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/ef/Youtube_logo.png" alt="Tutorial Video" width="24" height="24"/>
</a>


- Click the above button to open this notebook in Google Colab.
- We recommend then making a copy of the notebook to your Google Drive so you can save your work (File -> Save a copy in Drive).
- You may also view the accompanying supplamentary tutorial video [here](https://youtu.be/i4Sgg-ptRzs)

# Using OpenAI's Whisper for Transcription, Translation, and Creating Caption Files
OpenAI's Whisper is a general-purpose speech recognition model described in their 2022 [paper](https://arxiv.org/abs/2212.04356). This notebook is a practical introduction on how to use Whisper in Google Colab.

Before diving into Whisper, it's important to set up your environment correctly. This guide will walk you through the process, ensuring that even if you're not technically inclined, you'll be able to use Whisper effectively. Note that you don't need to run Whisper in Colab, we are using it here for convenience and the ability to run the model on a GPU.

## Setting Up Google Colab
Google Colab provides a convenient platform to run Python code in the cloud, with access to powerful computing resources, including GPUs. To ensure your Colab notebook runs smoothly, it's recommended to enable GPU acceleration which will speed up your transcription. Note however that a GPU is not strictly necessary to use Whisper. Here's how you can enable it on Colab:


1.   Click on 'Runtime' in the top menu.
2.   Select 'Change runtime type'.
3.   In the dialog that appears, under 'Hardware accelerator', choose 'GPU' (the type doesn't matter so much right now)
4.   Click 'Save'.

By enabling the GPU, your notebook will run more efficiently, especially when dealing with large models like Whisper.

We can also see the size of our GPU's VRAM with the `!nvidia-smi` command in order to see how large of a model we can use.

In [1]:
!nvidia-smi

Fri Nov  6 14:03:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Important Note

In the following cells, you will often see an `!` symbol before the text/commands. This is because Colab cells expect *code*.

By using `!`, we are telling Colab we typing a command instead of a piece of code! If you did not include a `!`, Colab would assume you are running (Python) code

In [2]:
print('Hello world!')

Hello world!


## Getting Started with Whisper

Whisper is available through OpenAI's GitHub repository. To use Whisper, you need to install it along with its dependencies. This guide will take you through the process step-by-step, ensuring a smooth setup. You may follow along in parallel looking at the Github as well! First, to install Whisper:



1.   **Install Whisper**: Run the command !pip install -U openai-whisper in a Colab cell to install the latest release of Whisper.
2.   **Install FFMPEG**: Whisper requires FFMPEG for audio processing. Use the command !apt install ffmpeg in Colab to install it.
3.   **Additional Dependencies:** In some cases, you might need additional dependencies like setuptools-rust. If you encounter any installation errors, run !pip install setuptools-rust.



In [1]:
!pip install -U openai-whisper
!apt install ffmpeg
!pip install setuptools-rust

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=09a3f71619c73e10d2f5fc059f6e66bf6d9cbe473179dd7c4d8645ae22510a24
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


## Available Models

Whisper comes with several models, each offering a trade-off between speed and accuracy. Depending on your task, you can choose the model that best fits your needs. Importantly, the *size* of the model (Required VRAM) will be dictated by the GPU running the code **if** you are using a GPU.

In this case, we will use the *medium* sized model.

## Get your audio files ready!

Of course, our goal is to transcripe/translation an audio file. Thus, we will need to upload our files to colab. This can be achieved through

1. Clicking the folder icon on the left sidebar at the bottom.
2. Clicking the upload file button or dragging and dropping your file into the left side bar.
3. For this demonstration, upload the `AllStar.mp3` file and `Cupid_Fifty_Fifty_Korean_Version.mp3` file from the GitHub [repository](https://github.com/keatonkraiger/Whisper-Transcribe-and-Translate-Tutorial).

**Be aware**: <ins>Colab sessions *will not* save files that have been uploaded or created. Be sure to save files that are created during sessions before exiting them.</ins>

**File Upload**: in the case you are uploading large files, you may instead want to mount your Google Drive to colab. You can then move files from Drive into Colab, which is often faster than uploading them individually. This can be achieved by clicking the Mount Drive button at the top left sidebar.

## Using Whisper

Once again, following the Github documentation, we can run Whisper through command-line arguments.

Let's first test whisper through the `whisper --help` command-line argument:

In [2]:
!whisper --help

usage: whisper [-h] [--model MODEL] [--model_dir MODEL_DIR] [--device DEVICE]
               [--output_dir OUTPUT_DIR]
               [--output_format {txt,vtt,srt,tsv,json,all}]
               [--verbose VERBOSE] [--task {transcribe,translate}]
               [--language {af,am,ar,as,az,ba,be,bg,bn,bo,br,bs,ca,cs,cy,da,de,el,en,es,et,eu,fa,fi,fo,fr,gl,gu,ha,haw,he,hi,hr,ht,hu,hy,id,is,it,ja,jw,ka,kk,km,kn,ko,la,lb,ln,lo,lt,lv,mg,mi,mk,ml,mn,mr,ms,mt,my,ne,nl,nn,no,oc,pa,pl,ps,pt,ro,ru,sa,sd,si,sk,sl,sn,so,sq,sr,su,sv,sw,ta,te,tg,th,tk,tl,tr,tt,uk,ur,uz,vi,yi,yo,yue,zh,Afrikaans,Albanian,Amharic,Arabic,Armenian,Assamese,Azerbaijani,Bashkir,Basque,Belarusian,Bengali,Bosnian,Breton,Bulgarian,Burmese,Cantonese,Castilian,Catalan,Chinese,Croatian,Czech,Danish,Dutch,English,Estonian,Faroese,Finnish,Flemish,French,Galician,Georgian,German,Greek,Gujarati,Haitian,Haitian Creole,Hausa,Hawaiian,Hebrew,Hindi,Hungarian,Icelandic,Indonesian,Italian,Japanese,Javanese,Kannada,Kazakh,Khmer,Korean,Lao,L

The above help command lists command-line arguments we can use. Most will be unnecessary for our case, while some may be more applicable such as `--output_format` or `--language`.

## Transcribing Audio

Let's now try and transcribe an audio file! The following command will transcribe speech in audio file `Audio File.mp3`. We will use the arguments

1.   `--model medium`: specify the model size
2.   `--task transcribe`: to do transcription
3.   `--output_dir transcription`: to save the files in the directory transcription
4.   `--output_format all`: give transcription in all formats. Otherwise, you may select the one your prefer.

Some caveats


*   If you are running this without a GPU, you should include: `--device cpu`
*   If you are interested in <ins>translating</ins> an audio file, you can use a command such as `!whisper japanese_audio_file.wav --language Japanese --task translate --model medium --output_dir translation --output_format all`


We must specify the location of the audio file. In this case it's in `/content/AllStar.mp3`. We can determine this by rightclicking the audiofile and Copying the Path.

In [11]:
!pip install -q gdown

import gdown

file_id = "1DovFJsb6x4fK32mcYnmcs5NxzF3Wi7Rb"

url = f"https://drive.google.com/uc?id={file_id}"

output = "/content/Property_Law.zip"

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1DovFJsb6x4fK32mcYnmcs5NxzF3Wi7Rb
From (redirected): https://drive.google.com/uc?id=1DovFJsb6x4fK32mcYnmcs5NxzF3Wi7Rb&confirm=t&uuid=1a93fb13-2378-4bf4-934a-252f31f4426d
To: /content/Property_Law.zip
100%|██████████| 1.17G/1.17G [00:13<00:00, 89.5MB/s]


'/content/Property_Law.zip'

In [12]:
import zipfile
import os

extract_path = "/content/Property_Law"

with zipfile.ZipFile("/content/Property_Law.zip", "r") as z:
    z.extractall(extract_path)

print("Extraction successful!")

Extraction successful!


In [13]:
audio_files = []

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(
            (".mp3", ".wav", ".m4a", ".flac", ".ogg")
        ):
            audio_files.append(os.path.join(root, file))

print("Total audio files:", len(audio_files))

Total audio files: 2000


In [16]:
!pip install -q faster-whisper

In [17]:
!nvidia-smi

Fri Nov  6 14:28:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "small",
    device="cuda",
    compute_type="float16"
)

print("Faster-Whisper loaded successfully on GPU!")

Faster-Whisper loaded successfully on GPU!


In [26]:
# Install required CUDA 12 library
!pip install -q nvidia-cublas-cu12

import os
import sys
import ctypes
import site

# Find the nvidia-cublas-cu12 lib path and pre-load the libraries into the active process
for site_dir in site.getsitepackages():
    cublas_path = os.path.join(site_dir, "nvidia", "cublas", "lib")
    if os.path.exists(cublas_path):
        # libcublasLt must be loaded before libcublas
        libcublaslt_path = os.path.join(cublas_path, "libcublasLt.so.12")
        libcublas_path = os.path.join(cublas_path, "libcublas.so.12")

        if os.path.exists(libcublaslt_path):
            ctypes.CDLL(libcublaslt_path, mode=ctypes.RTLD_GLOBAL)
        if os.path.exists(libcublas_path):
            ctypes.CDLL(libcublas_path, mode=ctypes.RTLD_GLOBAL)
        break

import av
import av.container.core
import time

# Safely reset av.open to the native C-function to destroy any recursive memory loops
av.open = av.container.core.open
av._original_open = av.container.core.open

# Apply the safe patch
def patched_open(*args, **kwargs):
    kwargs.pop('metadata_errors', None)
    return av._original_open(*args, **kwargs)

av.open = patched_open

test_audio = "/content/Property_Law/Property_Law/PROP-0001.wav"

start = time.time()

segments, info = model.transcribe(
    test_audio,
    task="transcribe",
    beam_size=1,
    vad_filter=True
)

segments = list(segments)

elapsed = time.time() - start

print("Detected language:", info.language)
print("Processing time:", round(elapsed, 2), "seconds")
print("Transcript:", " ".join(s.text for s in segments))

Detected language: en
Processing time: 0.93 seconds
Transcript:  Case Prop 0001 Issue whether the claimed property interest is recognized and sufficiently defined.  Argument whether the claimed property interest is recognized and sufficiently defined.  The governing principle should be.


In [27]:
import time

for task in ["transcribe", "translate"]:

    start = time.time()

    segments, info = model.transcribe(
        test_audio,
        task=task,
        beam_size=5,
        vad_filter=True
    )

    segments = list(segments)

    elapsed = time.time() - start

    print("\nTask:", task)
    print("Time:", round(elapsed, 2), "seconds")
    print("Text:", " ".join(s.text for s in segments))


Task: transcribe
Time: 0.59 seconds
Text:  Case Prop 0001. Issue whether the claimed property interest is recognized and sufficiently  defined. Argument whether the claimed property interest is recognized and sufficiently defined.  The governing principle should be.

Task: translate
Time: 0.5 seconds
Text:  Case Prop 0001. Issue whether the claimed property interest is recognized and sufficiently  defined. Argument whether the claimed property interest is recognized and sufficiently defined.  The governing principle should be.


In [29]:

# ============================================================
# PROPERTY LAW: RESUMABLE WHISPER TRANSCRIPTION + TRANSLATION
# Hardware: NVIDIA Tesla T4 (15 GB VRAM)
# Model: Faster-Whisper Small | FP16 | Batch size 8
# ============================================================

# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 2. Install dependencies
!pip -q install faster-whisper

import os
import csv
import json
import time
import torch
from pathlib import Path
from faster_whisper import WhisperModel, BatchedInferencePipeline

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

INPUT_FOLDER = Path("/content/Property_Law")

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Property_Law_Whisper_Output"
)

TRANSCRIPTION_DIR = OUTPUT_ROOT / "transcriptions"
TRANSLATION_DIR = OUTPUT_ROOT / "translations"

METADATA_FILE = OUTPUT_ROOT / "processing_metadata.csv"

MODEL_SIZE = "small"
COMPUTE_TYPE = "float16"

BATCH_SIZE = 8
BEAM_SIZE = 5
FILES_PER_BATCH = 100

AUDIO_EXTENSIONS = (
    ".mp3", ".wav", ".m4a", ".flac",
    ".ogg", ".mpeg", ".mpga", ".aac"
)

TRANSCRIPTION_DIR.mkdir(parents=True, exist_ok=True)
TRANSLATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. VERIFY GPU
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. Enable GPU in "
        "Runtime > Change runtime type."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

# ------------------------------------------------------------
# 4. LOAD MODEL ONCE
# ------------------------------------------------------------

print("\nLoading Faster-Whisper model...")

model = WhisperModel(
    MODEL_SIZE,
    device="cuda",
    compute_type=COMPUTE_TYPE
)

batched_model = BatchedInferencePipeline(model=model)

print("Model loaded successfully!")

# ------------------------------------------------------------
# 5. FIND AUDIO FILES
# ------------------------------------------------------------

if not INPUT_FOLDER.exists():
    raise FileNotFoundError(
        f"Input folder not found: {INPUT_FOLDER}"
    )

audio_files = sorted([
    p for p in INPUT_FOLDER.rglob("*")
    if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
])

print("\nTotal audio files found:", len(audio_files))

if not audio_files:
    raise RuntimeError("No supported audio files found.")

# ------------------------------------------------------------
# 6. METADATA / CHECKPOINT MANAGEMENT
# ------------------------------------------------------------

FIELDNAMES = [
    "audio_file",
    "source_language",
    "language_probability",
    "audio_duration_seconds",
    "transcription_status",
    "translation_status",
    "transcription_time_seconds",
    "translation_time_seconds",
    "total_processing_time_seconds",
    "overall_status",
    "error"
]

metadata = {}

if METADATA_FILE.exists():
    with open(METADATA_FILE, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            metadata[row["audio_file"]] = row


def save_metadata():
    """Rewrite metadata safely after each completed audio file."""
    temp_file = METADATA_FILE.with_suffix(".tmp")

    with open(temp_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()

        for key in sorted(metadata):
            writer.writerow(metadata[key])

    os.replace(temp_file, METADATA_FILE)


def save_json(data, path):
    """Write JSON atomically to avoid incomplete files."""
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    os.replace(temp_path, path)


def save_text(text, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


# ------------------------------------------------------------
# 7. PROCESSING FUNCTION
# ------------------------------------------------------------

def process_audio(audio_path):

    relative_path = audio_path.relative_to(INPUT_FOLDER)
    relative_stem = relative_path.with_suffix("")

    transcription_json = (
        TRANSCRIPTION_DIR / relative_stem.with_suffix(".json")
    )
    transcription_txt = (
        TRANSCRIPTION_DIR / relative_stem.with_suffix(".txt")
    )

    translation_json = (
        TRANSLATION_DIR / relative_stem.with_suffix(".json")
    )
    translation_txt = (
        TRANSLATION_DIR / relative_stem.with_suffix(".txt")
    )

    record = metadata.get(str(relative_path), {}).copy()

    record.setdefault("audio_file", str(relative_path))
    record.setdefault("source_language", "")
    record.setdefault("language_probability", "")
    record.setdefault("audio_duration_seconds", "")
    record.setdefault("transcription_status", "pending")
    record.setdefault("translation_status", "pending")
    record.setdefault("transcription_time_seconds", "")
    record.setdefault("translation_time_seconds", "")
    record.setdefault("total_processing_time_seconds", "")
    record.setdefault("error", "")

    total_start = time.perf_counter()
    errors = []

    source_language = record.get("source_language") or None

    # --------------------------------------------------------
    # TRANSCRIPTION
    # --------------------------------------------------------

    if transcription_json.exists() and transcription_txt.exists():
        record["transcription_status"] = "completed"
        print("  Transcription already exists - skipped")

    else:
        try:
            start = time.perf_counter()

            segments, info = batched_model.transcribe(
                str(audio_path),
                task="transcribe",
                batch_size=BATCH_SIZE,
                beam_size=BEAM_SIZE,
                vad_filter=True
            )

            segments = list(segments)

            transcript = " ".join(
                segment.text.strip() for segment in segments
            )

            transcription_data = {
                "audio_file": str(relative_path),
                "task": "transcribe",
                "language": info.language,
                "language_probability": info.language_probability,
                "audio_duration_seconds": getattr(info, "duration", None),
                "segments": [
                    {
                        "start": segment.start,
                        "end": segment.end,
                        "text": segment.text.strip()
                    }
                    for segment in segments
                ]
            }

            save_json(transcription_data, transcription_json)
            save_text(transcript, transcription_txt)

            source_language = info.language

            record["source_language"] = info.language
            record["language_probability"] = info.language_probability
            record["audio_duration_seconds"] = getattr(
                info, "duration", ""
            )
            record["transcription_status"] = "completed"
            record["transcription_time_seconds"] = round(
                time.perf_counter() - start, 3
            )

            print(
                "  Transcription completed:",
                record["transcription_time_seconds"], "seconds"
            )

        except Exception as e:
            record["transcription_status"] = "failed"
            errors.append("Transcription: " + str(e))
            print("  Transcription failed:", e)

    # --------------------------------------------------------
    # TRANSLATION
    # --------------------------------------------------------

    if translation_json.exists() and translation_txt.exists():
        record["translation_status"] = "completed"
        print("  Translation already exists - skipped")

    else:
        try:
            start = time.perf_counter()

            # Reuse detected source language where available.
            segments, info = batched_model.transcribe(
                str(audio_path),
                task="translate",
                language=source_language,
                batch_size=BATCH_SIZE,
                beam_size=BEAM_SIZE,
                vad_filter=True
            )

            segments = list(segments)

            translated_text = " ".join(
                segment.text.strip() for segment in segments
            )

            translation_data = {
                "audio_file": str(relative_path),
                "task": "translate",
                "source_language": source_language or info.language,
                "target_language": "English",
                "segments": [
                    {
                        "start": segment.start,
                        "end": segment.end,
                        "text": segment.text.strip()
                    }
                    for segment in segments
                ]
            }

            save_json(translation_data, translation_json)
            save_text(translated_text, translation_txt)

            record["translation_status"] = "completed"
            record["translation_time_seconds"] = round(
                time.perf_counter() - start, 3
            )

            print(
                "  Translation completed:",
                record["translation_time_seconds"], "seconds"
            )

        except Exception as e:
            record["translation_status"] = "failed"
            errors.append("Translation: " + str(e))
            print("  Translation failed:", e)

    # --------------------------------------------------------
    # SAVE CHECKPOINT
    # --------------------------------------------------------

    record["total_processing_time_seconds"] = round(
        time.perf_counter() - total_start, 3
    )

    record["error"] = " | ".join(errors)

    if (
        record["transcription_status"] == "completed"
        and record["translation_status"] == "completed"
    ):
        record["overall_status"] = "completed"
    elif (
        record["transcription_status"] == "failed"
        and record["translation_status"] == "failed"
    ):
        record["overall_status"] = "failed"
    else:
        record["overall_status"] = "partial"

    metadata[str(relative_path)] = record
    save_metadata()

    return record


# ------------------------------------------------------------
# 8. PROCESS IN GROUPS OF 100
# ------------------------------------------------------------

overall_start = time.perf_counter()

completed_count = 0
failed_count = 0
skipped_count = 0

total_files = len(audio_files)

for batch_start in range(0, total_files, FILES_PER_BATCH):

    batch_number = batch_start // FILES_PER_BATCH + 1
    batch_files = audio_files[
        batch_start:batch_start + FILES_PER_BATCH
    ]

    print("\n" + "=" * 60)
    print(
        f"BATCH {batch_number} | "
        f"Files {batch_start + 1} to "
        f"{batch_start + len(batch_files)} of {total_files}"
    )
    print("=" * 60)

    for index, audio_path in enumerate(batch_files):

        global_index = batch_start + index + 1

        relative_path = str(audio_path.relative_to(INPUT_FOLDER))

        print(
            f"\n[{global_index}/{total_files}] {relative_path}"
        )

        trans_path = (
            TRANSCRIPTION_DIR
            / Path(relative_path).with_suffix(".json")
        )
        trans_txt = (
            TRANSCRIPTION_DIR
            / Path(relative_path).with_suffix(".txt")
        )
        translate_path = (
            TRANSLATION_DIR
            / Path(relative_path).with_suffix(".json")
        )
        translate_txt = (
            TRANSLATION_DIR
            / Path(relative_path).with_suffix(".txt")
        )

        if all(p.exists() for p in [
            trans_path, trans_txt, translate_path, translate_txt
        ]):
            skipped_count += 1
            print("Already completed - skipped")
            continue

        result = process_audio(audio_path)

        if result["overall_status"] == "completed":
            completed_count += 1
        else:
            failed_count += 1

    print(f"\nBatch {batch_number} checkpoint saved.")
    print("Completed:", completed_count)
    print("Failed/partial:", failed_count)
    print("Skipped:", skipped_count)

# ------------------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------------------

total_elapsed = time.perf_counter() - overall_start

print("\n" + "=" * 60)
print("PROCESSING FINISHED")
print("=" * 60)

print("Total audio files:", total_files)
print("Completed during this run:", completed_count)
print("Failed/partial:", failed_count)
print("Already completed/skipped:", skipped_count)
print("Total elapsed minutes:", round(total_elapsed / 60, 2))

print("\nOutput location:", OUTPUT_ROOT)
print("Metadata CSV:", METADATA_FILE)


Streaming output truncated to the last 5000 lines.
[782/2000] Property_Law/PROP-0782.wav
  Transcription completed: 0.626 seconds
  Translation completed: 0.547 seconds

[783/2000] Property_Law/PROP-0783.wav
  Transcription completed: 0.645 seconds
  Translation completed: 0.75 seconds

[784/2000] Property_Law/PROP-0784.wav
  Transcription completed: 0.599 seconds
  Translation completed: 0.603 seconds

[785/2000] Property_Law/PROP-0785.wav
  Transcription completed: 0.591 seconds
  Translation completed: 0.388 seconds

[786/2000] Property_Law/PROP-0786.wav
  Transcription completed: 0.502 seconds
  Translation completed: 0.455 seconds

[787/2000] Property_Law/PROP-0787.wav
  Transcription completed: 0.524 seconds
  Translation completed: 0.457 seconds

[788/2000] Property_Law/PROP-0788.wav
  Transcription completed: 0.52 seconds
  Translation completed: 0.453 seconds

[789/2000] Property_Law/PROP-0789.wav
  Transcription completed: 0.531 seconds
  Translation completed: 0.439 seconds


In [30]:
!git clone https://github.com/hari-hud/ai-metrics.git

%cd ai-metrics

!pip install -r requirements.txt

!pip install -e .

Cloning into 'ai-metrics'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 42 (delta 4), reused 38 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 17.22 KiB | 4.31 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/ai-metrics
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=2d3f1960473948bc9a8290904dace98151910ab55b31ef30f4db3cfc6c2f0524
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [bert_score]
Obtaining file:///content/ai-metrics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting req

In [31]:
from metrics.nlp.bert_score import BertScore
from metrics.nlp.bleu_score import BLEUScore
from metrics.nlp.rouge_score import ROUGEScores

# Gold-standard sentence
ground_truths = [
    "The tenant has the right to occupy the property because a valid lease agreement exists between the landlord and the tenant."
]

# Whisper-generated sentence
predictions = [
    "The tenant has a right to occupy the property because a valid rental agreement exists between the landlord and tenant."
]

# BERTScore
bert = BertScore()
bert_result = bert.get_score(ground_truths, predictions)

# BLEU
bleu = BLEUScore()
bleu_result = bleu.get_score(ground_truths, predictions)

# ROUGE
rouge = ROUGEScores()
rouge_result = rouge.get_score(ground_truths, predictions)

# Display results
print("BERTScore:", bert_result)
print("BLEU Score:", bleu_result)
print("ROUGE Score:", rouge_result)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore: {'bert_f1': 0.9664638042449951}
BLEU Score: {'bleu_score': 61.6357016548165}
ROUGE Score: {'rouge_1_score': 0.8780487804878048, 'rouge_2_score': 0.717948717948718, 'rouge_3_score': 0.5945945945945946, 'rouge_L_score': 0.8780487804878048}


In [34]:
import pandas as pd
import os

from metrics.nlp.bert_score import BertScore
from metrics.nlp.bleu_score import BLEUScore
from metrics.nlp.rouge_score import ROUGEScores

eval_file = "whisper_evaluation.csv"

if not os.path.exists(eval_file):
    print(f"❌ Error: '{eval_file}' was not found in the current directory.")
    print("\nHow to fix this:")
    print("1. Make sure you have created or uploaded a CSV named 'whisper_evaluation.csv'.")
    print("2. The CSV should contain at least two columns:")
    print("   - 'Gold Standard' (the manually verified correct text)")
    print("   - 'Whisper Generated' (the text generated by the model)")
    print("\nExample of how you can build this dataframe from your output:")
    print("""# Example Code to assemble your evaluation dataset:
# df = pd.DataFrame({
#     "Gold Standard": ["The tenant has the right to occupy the property because a valid lease agreement exists..."],
#     "Whisper Generated": ["The tenant has a right to occupy the property because a valid rental agreement exists..."]
# })
# df.to_csv("whisper_evaluation.csv", index=False)""")
else:
    # Load dataset
    df = pd.read_csv(eval_file)

    # Extract reference and generated sentences
    ground_truths = df["Gold Standard"].fillna("").astype(str).tolist()
    predictions = df["Whisper Generated"].fillna("").astype(str).tolist()

    # Initialize metrics
    bert = BertScore()
    bleu = BLEUScore()
    rouge = ROUGEScores()

    # Calculate metrics
    bert_result = bert.get_score(ground_truths, predictions)
    bleu_result = bleu.get_score(ground_truths, predictions)
    rouge_result = rouge.get_score(ground_truths, predictions)

    # Display results
    print("\n===== WHISPER EVALUATION =====")
    print("BERTScore:", bert_result)
    print("BLEU:", bleu_result)
    print("ROUGE:", rouge_result)

❌ Error: 'whisper_evaluation.csv' was not found in the current directory.

How to fix this:
1. Make sure you have created or uploaded a CSV named 'whisper_evaluation.csv'.
2. The CSV should contain at least two columns:
   - 'Gold Standard' (the manually verified correct text)
   - 'Whisper Generated' (the text generated by the model)

Example of how you can build this dataframe from your output:
# Example Code to assemble your evaluation dataset:
# df = pd.DataFrame({
#     "Gold Standard": ["The tenant has the right to occupy the property because a valid lease agreement exists..."],
#     "Whisper Generated": ["The tenant has a right to occupy the property because a valid rental agreement exists..."]
# })
# df.to_csv("whisper_evaluation.csv", index=False)


In [28]:
import soundfile as sf

audio, sr = sf.read(test_audio)
duration = len(audio) / sr

print("Audio duration:", round(duration / 60, 2), "minutes")

Audio duration: 0.24 minutes


In [15]:
import subprocess
import os

# Define output folders if they aren't already set up
transcription_folder = "/content/transcription"
translation_folder = "/content/translation"

os.makedirs(transcription_folder, exist_ok=True)
os.makedirs(translation_folder, exist_ok=True)

for i, audio in enumerate(audio_files):
    print(f"\nProcessing {i+1}/{len(audio_files)}: {os.path.basename(audio)}")

    subprocess.run([
        "whisper", audio,
        "--model", "medium",
        "--task", "transcribe",
        "--output_dir", transcription_folder,
        "--output_format", "all"
    ], check=True)

    subprocess.run([
        "whisper", audio,
        "--model", "medium",
        "--task", "translate",
        "--output_dir", translation_folder,
        "--output_format", "all"
    ], check=True)

print("All audio files processed!")


Processing 1/2000: PROP-0320.wav

Processing 2/2000: PROP-0300.wav

Processing 3/2000: PROP-1631.wav

Processing 4/2000: PROP-0093.wav

Processing 5/2000: PROP-1203.wav

Processing 6/2000: PROP-1912.wav

Processing 7/2000: PROP-1977.wav

Processing 8/2000: PROP-1610.wav

Processing 9/2000: PROP-0297.wav

Processing 10/2000: PROP-1105.wav

Processing 11/2000: PROP-1540.wav

Processing 12/2000: PROP-0907.wav


KeyboardInterrupt: 

In [3]:
!whisper /content/AllStar.mp3 --model medium --task transcribe --output_dir transcription --output_format all

100%|█████████████████████████████████████| 1.42G/1.42G [01:40<00:00, 15.2MiB/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/audio.py", line 58, in load_audio
    out = run(cmd, capture_output=True, check=True).stdout
          ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 578, in run
    raise CalledProcessError(retcode, process.args,
                             output=stdout, stderr=stderr)
subprocess.CalledProcessError: Command '['ffmpeg', '-nostdin', '-threads', '0', '-i', '/content/AllStar.mp3', '-f', 's16le', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '16000', '-']' returned non-zero exit status 254.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 615, in cli
    result = transcribe(model, audio_path, temperature=temperature, **args)
  File "/usr/local/lib

**Note**:
- Most of the time you won't want to use the `all` output format as it will generate some unnecessary files. Its best to determine which format works for you. Recall, you can find the available formats by running `whisper --help`.
- If you are running this on your own computer, the path for the file will be different. You can find the path by right-clicking the file and selecting "Copy Path".

## Translating Audio

Let's now try and translate an audio file! Make sure you uploaded the `Cupid_Fifty_Fifty_Korean_Version.mp3` file to Colab. To see which languages Whisper supports, you can run `!whisper --help` and look at the `--language` argument.

In [4]:
!whisper --help

usage: whisper [-h] [--model MODEL] [--model_dir MODEL_DIR] [--device DEVICE]
               [--output_dir OUTPUT_DIR]
               [--output_format {txt,vtt,srt,tsv,json,all}]
               [--verbose VERBOSE] [--task {transcribe,translate}]
               [--language {af,am,ar,as,az,ba,be,bg,bn,bo,br,bs,ca,cs,cy,da,de,el,en,es,et,eu,fa,fi,fo,fr,gl,gu,ha,haw,he,hi,hr,ht,hu,hy,id,is,it,ja,jw,ka,kk,km,kn,ko,la,lb,ln,lo,lt,lv,mg,mi,mk,ml,mn,mr,ms,mt,my,ne,nl,nn,no,oc,pa,pl,ps,pt,ro,ru,sa,sd,si,sk,sl,sn,so,sq,sr,su,sv,sw,ta,te,tg,th,tk,tl,tr,tt,uk,ur,uz,vi,yi,yo,yue,zh,Afrikaans,Albanian,Amharic,Arabic,Armenian,Assamese,Azerbaijani,Bashkir,Basque,Belarusian,Bengali,Bosnian,Breton,Bulgarian,Burmese,Cantonese,Castilian,Catalan,Chinese,Croatian,Czech,Danish,Dutch,English,Estonian,Faroese,Finnish,Flemish,French,Galician,Georgian,German,Greek,Gujarati,Haitian,Haitian Creole,Hausa,Hawaiian,Hebrew,Hindi,Hungarian,Icelandic,Indonesian,Italian,Japanese,Javanese,Kannada,Kazakh,Khmer,Korean,Lao,L

Suppose we have a Korean audio file and we want to translate it to English. We can use the following command:

In [5]:
!whisper /content/Cupid_Fifty_Fifty_Korean_Version.mp3 --language Korean --task translate --model medium --output_dir translation --output_format all

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/audio.py", line 58, in load_audio
    out = run(cmd, capture_output=True, check=True).stdout
          ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 578, in run
    raise CalledProcessError(retcode, process.args,
                             output=stdout, stderr=stderr)
subprocess.CalledProcessError: Command '['ffmpeg', '-nostdin', '-threads', '0', '-i', '/content/Cupid_Fifty_Fifty_Korean_Version.mp3', '-f', 's16le', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '16000', '-']' returned non-zero exit status 254.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 615, in cli
    result = transcribe(model, audio_path, temperature=temperature, **args)
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 1

## Creating a Caption File (SRT)

Suppose we want to take an audio file and create a caption file (SRT) for it. SRT files are used to display subtitles in videos and are commonly used in video editing software or on YouTube. Let's try creating an SRT file for an audio file. We will use the following command:

In [6]:
!whisper /content/AllStar.mp3 --model medium --task transcribe --output_dir transcription --output_format srt

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/audio.py", line 58, in load_audio
    out = run(cmd, capture_output=True, check=True).stdout
          ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 578, in run
    raise CalledProcessError(retcode, process.args,
                             output=stdout, stderr=stderr)
subprocess.CalledProcessError: Command '['ffmpeg', '-nostdin', '-threads', '0', '-i', '/content/AllStar.mp3', '-f', 's16le', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '16000', '-']' returned non-zero exit status 254.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 615, in cli
    result = transcribe(model, audio_path, temperature=temperature, **args)
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 139, in transcribe
    mel

What if the audio is in a different language? We can use the `--language` argument to specify the language of the audio file. For example, if the audio file is in Korean, we can use the following command:

In [7]:
!whisper /content/Cupid_Fifty_Fifty_Korean_Version.mp3 --language Korean --task translate --model medium --output_dir translation --output_format srt

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/audio.py", line 58, in load_audio
    out = run(cmd, capture_output=True, check=True).stdout
          ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 578, in run
    raise CalledProcessError(retcode, process.args,
                             output=stdout, stderr=stderr)
subprocess.CalledProcessError: Command '['ffmpeg', '-nostdin', '-threads', '0', '-i', '/content/Cupid_Fifty_Fifty_Korean_Version.mp3', '-f', 's16le', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '16000', '-']' returned non-zero exit status 254.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 615, in cli
    result = transcribe(model, audio_path, temperature=temperature, **args)
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 1

## Saving Your Files

Once you've ran all the cells you're interested in, we should download the transcription files that are saved to the `transcription` directory.

Colab doesn't allow you to download entire directories so we can instead `zip` it and download the zip file.

In [8]:
!zip -r transcriptions.zip transcription
!zip -r translations.zip translation

  adding: transcription/ (stored 0%)
  adding: translation/ (stored 0%)


## Helpful Commands



*   Often, you fill note want the transcription in every format. It may instead be better to select the most usable one for your case and just use that as the `--output_format`
*   You can do multiple audio files at a time. You would do the same command, but include two or more audio files (e.g. `!whisper audio1.mp3 audio2.mp3 ...`)
* If you are running whisper without a GPU, you should include `--device cpu` in the command
* If you want each caption segment to be 3 words max opposed to sentences you could do something like
```bash
  whisper audio_file.mp3 --task transcribe --output_format srt --word_timestamps True --max_words_per_line 3 # for transcription`
```


## Closing Remarks

Using Whisper in Colab is convenient because it allows you to utilize a GPU for free! However, you can also use Whisper without a GPU on your local computer. To do this, you should follow the steps in Whisper's [Github Setup](https://github.com/openai/whisper#setup). If you choose to do so, some considerations



1.   You will need to install Python to run whisper. This can be done in your terminal by first installing [brew](https://docs.brew.sh/Installation) with the command `/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"`. We recommend following `HomeBrew's [installation instructions.](https://brew.sh/) or this [guide](https://docs.python-guide.org/starting/install3/osx/).
2.   Assuming brew is installed, you can install python with running `brew install python` inside your terminal.
3.   With Python and brew installed, we recommend making a directory to work in. Inside your terminal, move to your desktop and create a directory: `cd Desktop; mkdir Whisper; cd Whisper`.

Note: if you do wish to work on your personal macbook and do install brew, you will need to also install `Xcode` tools. If given the option, we recommend instead installing `Command Line Tools` as it is a much smaller package.

Finally, while this document is primarily for mac users, the installation will be very similar across platforms. See Whisper's Github for more information.

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
     <img src="https://media.istockphoto.com/id/1294688589/photo/red-cat-with-blurred-the-poster-in-the-frame-with-the-words-thank-you.jpg?s=612x612&w=0&k=20&c=T84nHSu52sOQvrmnksdDNo2UByqJ7yXn1srkuodXdps=" alt="Page Image">
    <br>
</body>
</html>

### Step 1: Unzip your audio files
Upload your `.zip` file containing your audio tracks to Google Colab using the file explorer sidebar on the left. Then, replace `your_file.zip` below with your actual zip filename and run the cell to extract them.

In [9]:
# Extract the zipped audio files into a folder named 'extracted_audio'
!unzip -q /content/your_file.zip -d /content/extracted_audio

# List the extracted files to verify
!ls /content/extracted_audio

unzip:  cannot find or open /content/your_file.zip, /content/your_file.zip.zip or /content/your_file.zip.ZIP.
ls: cannot access '/content/extracted_audio': No such file or directory


### Step 2: Run Whisper on the extracted files
Now you can run Whisper on one of the extracted files. Replace `audio_filename.mp3` with the name of the file you want to transcribe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:
!whisper /content/extracted_audio/audio_filename.mp3 --model medium --task transcribe --output_dir transcription --output_format all

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/audio.py", line 58, in load_audio
    out = run(cmd, capture_output=True, check=True).stdout
          ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 578, in run
    raise CalledProcessError(retcode, process.args,
                             output=stdout, stderr=stderr)
subprocess.CalledProcessError: Command '['ffmpeg', '-nostdin', '-threads', '0', '-i', '/content/extracted_audio/audio_filename.mp3', '-f', 's16le', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '16000', '-']' returned non-zero exit status 254.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 615, in cli
    result = transcribe(model, audio_path, temperature=temperature, **args)
  File "/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py", line 139